# rfahrur6045-testing.ipynb
## Testing & Prediction Request ke Cloud (Railway)

Notebook ini digunakan untuk menguji sistem machine learning yang telah di-deploy di Railway dan melakukan prediction request ke TF Serving endpoint.

**Endpoint:** `https://<railway-url>/v1/models/ecommerce_churn:predict`

## 1. Setup

In [ ]:
import requests
import json
import pandas as pd
import numpy as np

# ── Ganti dengan URL Railway Anda ──────────────────────────────
RAILWAY_URL = "https://rfahrur6045-churn-serving.up.railway.app"
PREDICT_URL = f"{RAILWAY_URL}/v1/models/ecommerce_churn:predict"
STATUS_URL  = f"{RAILWAY_URL}/v1/models/ecommerce_churn"

print(f"Predict URL : {PREDICT_URL}")
print(f"Status URL  : {STATUS_URL}")

## 2. Cek Status Model di Cloud

In [ ]:
response = requests.get(STATUS_URL)
print(f"Status Code : {response.status_code}")
print(f"Response    : {json.dumps(response.json(), indent=2)}")

## 3. Definisi Sample Input

Kita siapkan beberapa skenario pelanggan untuk diuji:
- **High Risk**: Pelanggan baru, banyak komplain, skor kepuasan rendah
- **Low Risk**: Pelanggan lama, tidak pernah komplain, skor kepuasan tinggi

In [ ]:
# ── Skenario 1: High Risk Churn ────────────────────────────────
high_risk = {
    "Tenure": 2,
    "CityTier": 1,
    "WarehouseToHome": 45.0,
    "HourSpendOnApp": 1.0,
    "NumberOfDeviceRegistered": 6,
    "PreferredLoginDevice": "Mobile Phone",
    "PreferredPaymentMode": "Cash on Delivery",
    "Gender": "Male",
    "PreferedOrderCat": "Mobile Phone",
    "SatisfactionScore": 1,
    "MaritalStatus": "Single",
    "NumberOfAddress": 9,
    "Complain": 1,
    "OrderAmountHikeFromlastYear": 11.0,
    "CouponUsed": 1.0,
    "OrderCount": 2.0,
    "DaySinceLastOrder": 15.0,
    "CashbackAmount": 120.9
}

# ── Skenario 2: Low Risk Churn ─────────────────────────────────
low_risk = {
    "Tenure": 24,
    "CityTier": 1,
    "WarehouseToHome": 10.0,
    "HourSpendOnApp": 5.0,
    "NumberOfDeviceRegistered": 2,
    "PreferredLoginDevice": "Computer",
    "PreferredPaymentMode": "Credit Card",
    "Gender": "Female",
    "PreferedOrderCat": "Laptop & Accessory",
    "SatisfactionScore": 5,
    "MaritalStatus": "Married",
    "NumberOfAddress": 2,
    "Complain": 0,
    "OrderAmountHikeFromlastYear": 25.0,
    "CouponUsed": 8.0,
    "OrderCount": 15.0,
    "DaySinceLastOrder": 2.0,
    "CashbackAmount": 280.0
}

print("Sample data siap!")

## 4. Fungsi Prediction Request

In [ ]:
def predict_churn(instance: dict, url: str = PREDICT_URL) -> dict:
    """
    Mengirim prediction request ke TF Serving endpoint.
    
    Args:
        instance: dict berisi fitur pelanggan.
        url: URL endpoint TF Serving.
    
    Returns:
        dict berisi probabilitas churn dan label prediksi.
    """
    payload = {"instances": [instance]}
    
    response = requests.post(
        url,
        headers={"Content-Type": "application/json"},
        data=json.dumps(payload),
        timeout=30
    )
    
    if response.status_code != 200:
        raise ValueError(f"Error {response.status_code}: {response.text}")
    
    prob = response.json()["predictions"][0][0]
    label = "CHURN" if prob >= 0.5 else "TIDAK CHURN"
    risk = "🔴 High Risk" if prob >= 0.7 else ("🟡 Medium Risk" if prob >= 0.5 else "🟢 Low Risk")
    
    return {
        "probabilitas_churn": round(prob, 4),
        "prediksi": label,
        "risk_level": risk
    }

print("Fungsi predict_churn siap!")

## 5. Uji Prediksi - Skenario High Risk

In [ ]:
print("=" * 50)
print("SKENARIO: Pelanggan High Risk")
print("=" * 50)
result = predict_churn(high_risk)
for k, v in result.items():
    print(f"  {k}: {v}")

## 6. Uji Prediksi - Skenario Low Risk

In [ ]:
print("=" * 50)
print("SKENARIO: Pelanggan Low Risk")
print("=" * 50)
result = predict_churn(low_risk)
for k, v in result.items():
    print(f"  {k}: {v}")

## 7. Batch Prediction - 10 Sample Acak

In [ ]:
import random

login_devices   = ['Mobile Phone', 'Computer', 'Phone']
payment_modes   = ['Debit Card', 'UPI', 'Credit Card', 'Cash on Delivery', 'E wallet']
genders         = ['Male', 'Female']
order_cats      = ['Laptop & Accessory', 'Mobile', 'Mobile Phone', 'Others', 'Fashion', 'Grocery']
marital_statuses = ['Single', 'Divorced', 'Married']

results = []
for i in range(10):
    sample = {
        "Tenure": random.randint(0, 60),
        "CityTier": random.choice([1, 2, 3]),
        "WarehouseToHome": float(random.randint(5, 127)),
        "HourSpendOnApp": float(random.choice([0, 1, 2, 3, 4, 5])),
        "NumberOfDeviceRegistered": random.randint(1, 6),
        "PreferredLoginDevice": random.choice(login_devices),
        "PreferredPaymentMode": random.choice(payment_modes),
        "Gender": random.choice(genders),
        "PreferedOrderCat": random.choice(order_cats),
        "SatisfactionScore": random.randint(1, 5),
        "MaritalStatus": random.choice(marital_statuses),
        "NumberOfAddress": random.randint(1, 22),
        "Complain": random.choice([0, 1]),
        "OrderAmountHikeFromlastYear": float(random.randint(11, 26)),
        "CouponUsed": float(random.randint(0, 15)),
        "OrderCount": float(random.randint(1, 16)),
        "DaySinceLastOrder": float(random.randint(0, 45)),
        "CashbackAmount": round(random.uniform(0, 325), 2)
    }
    result = predict_churn(sample)
    result['sample_id'] = i + 1
    results.append(result)

df_results = pd.DataFrame(results)[['sample_id', 'probabilitas_churn', 'prediksi', 'risk_level']]
print(df_results.to_string(index=False))
print(f"\nTotal prediksi CHURN : {(df_results['prediksi'] == 'CHURN').sum()} dari 10 sampel")

## 8. Kesimpulan Testing

Sistem machine learning telah berhasil diuji di environment cloud (Railway):
- ✅ Model status: `AVAILABLE`
- ✅ REST API endpoint merespons dengan benar
- ✅ Prediksi High Risk menunjukkan probabilitas churn tinggi
- ✅ Prediksi Low Risk menunjukkan probabilitas churn rendah
- ✅ Batch prediction dari 10 sampel berjalan normal